# COS30049 A2 - Round 2: the 152M model's chat and explanation stages, then every evaluation (T4 x2)

After the continued pretraining (`cos30049a2-text150m-train`), round 1's two fine-tuning stages run again from the
new weights on both GPUs: chat on DS6 (`configs/text_150m_chat_v2.json`), then explanations on DS7 v2
(`configs/text_150m_explain_v2.json`). The new writer is then scored on the frozen DS7 v1 test, DS10 and the glossary
test, and round 1's writer on the glossary test, side by side on the two GPUs (`docs/KAGGLE_LM_PLAN.md` section 3).

**Settings:** Accelerator **GPU T4 x2**, **Internet on**. **Inputs:** this account's dataset `cos30049a2-text150m-base`
and notebooks `cos30049a2-text150m-stage-data` and `cos30049a2-text150m-train` (its finished `model.pt`); to resume,
also this notebook's own previous output, with `RESUME_FROM` set.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "028ec4c2209f784ec2abba49cc1a5f73a59e7cd6"   # the code this run uses; recorded in kaggle_run.json
SESSION_HOURS = 11.75  # Kaggle ends a GPU session at 12 h
EVAL_HOURS = 0.75      # what the evaluations need on two GPUs
RESUME_FROM = None     # e.g. "/kaggle/input/<this notebook>/runs" to continue a stopped session
PINS = ["tokenizers==0.21.1", "numpy==2.2.6"]  # environment.yml; torch is Kaggle's own (recorded)

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
RUNS = WORK / "runs"

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(GPUS)]

In [ ]:
base = check_base()
stage_manifest_path = find_input("stage_manifest.json")
for rel, digest in json.loads(stage_manifest_path.read_text(encoding="utf-8"))["files"].items():
    assert sha256(stage_manifest_path.parent / rel) == digest, f"{rel} differs from stage_manifest.json"
STAGE = stage_manifest_path.parent / "stage"
PRETRAINED = find_input("runs/text-150m-cont/model.pt")
if RESUME_FROM and not RUNS.exists():
    shutil.copytree(RESUME_FROM, RUNS)
    print("resuming from", RESUME_FROM)


def config(name, **paths):
    cfg = json.loads((SRC / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    cfg.update({k: str(v) for k, v in paths.items()})
    path = Path(f"/tmp/{name}.json")
    path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return path


CHAT, EXPLAIN = RUNS / "text-150m-chat-v2", RUNS / "text-150m-explain-v2"
STAGES = [(config("text_150m_chat_v2", data_dir=STAGE / "chat", out_dir=CHAT, init_from=PRETRAINED), CHAT),
          (config("text_150m_explain_v2", data_dir=STAGE / "ds7_v2_chat", out_dir=EXPLAIN,
                  init_from=CHAT / "model.pt"), EXPLAIN)]

In [ ]:
for cfg, out in STAGES:
    if (out / "model.pt").exists():
        continue
    hours = SESSION_HOURS - (time.time() - T0) / 3600 - EVAL_HOURS - 0.2
    sh(*TORCHRUN, "-m", "lm.train", "--config", str(cfg), "--max-hours", f"{hours:.2f}", log=f"{out.name}.log")
    if not (out / "model.pt").exists():
        print(f"{out.name} stopped for time: run again with RESUME_FROM (see the top)")
        break

In [ ]:
# Every evaluation the adoption rule needs, one queue per GPU; each test set writes its own tables.
V2 = EXPLAIN / "model_best.pt"
GLOSSARY = STAGE / "glossary_test.jsonl"
RANDOM = SRC / "data" / "processed" / "random_test"
FIRST = [("text150m_v2", V2, SRC / "data" / "processed" / "ds7" / "test.jsonl", "")] + [
    ("text150m_v2", V2, RANDOM / f"{part}.jsonl", f"random_{part}") for part in ("real", "hard", "crawl_ds1", "crawl_ds10", "extreme")]
SECOND = [("text150m_v2", V2, SRC / "data" / "processed" / "ds10" / "explain.jsonl", "ds10"),
          ("text150m_v2", V2, GLOSSARY, "glossary"),
          ("text150m", base / "text150m_explain_v1.pt", GLOSSARY, "glossary")] + [
    ("text150m_v2", V2, RANDOM / f"{part}.jsonl", f"random_{part}")
    for part in ("synthetic", "balanced", "crawl_ds2", "close_calls", "max_shift")]
# A test set's evaluations stay in one queue: they write one summary table.
QUEUES = [FIRST, SECOND] if GPUS > 1 else [FIRST + SECOND]


def run_queue(gpu, queue):
    for model, checkpoint, test, tag in queue:
        # --n above every set's size: each prompt is answered (the default draws 500)
        sh(sys.executable, "-m", "lm.eval_explain", "--model", model, "--checkpoint", str(checkpoint), "--test", str(test),
           "--tag", tag, "--n", "100000", log=f"eval-{model}-{tag or 'ds7'}.log",
           env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))


if (EXPLAIN / "model.pt").exists():
    threads = [threading.Thread(target=run_queue, args=(gpu, q)) for gpu, q in enumerate(QUEUES)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    tables = WORK / "reports" / "tables"
    tables.mkdir(parents=True, exist_ok=True)
    for f in (SRC / "reports" / "tables").glob("assistant_eval*.csv"):
        shutil.copy(f, tables)
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "gpus": GPUS, "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "numpy", "tokenizers")}, "resumed_from": RESUME_FROM,
    "session_hours": round((time.time() - T0) / 3600, 2),
    "finished": {out.name: (out / "model.pt").exists() for _, out in STAGES},
}, indent=2), encoding="utf-8")